# 01 Test Bronze Access & Relationship Validation

**Project:** Metro Vancouver Transit Analytics  
**Storage Account:** `stransitanalytics`  
**Container:** `bronze`  

This notebook verifies:
1. Synapse Spark connectivity to ADLS Gen2 `stransitanalytics`
2. Reading the four primary static GTFS datasets (`routes.txt`, `trips.txt`, `stops.txt`, `stop_times.txt`)
3. Accurate row count checks against source datasets
4. Foreign key integrity and orphan checks across tables

In [ ]:
# Phase 6: Connect to ADLS Gen2 Bronze and Read routes.txt
storage_account = "stransitanalytics"

routes_path = (
    f"abfss://bronze@{storage_account}.dfs.core.windows.net/"
    "gtfs_static/routes.txt"
)

routes_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(routes_path)
)

print("Rows:", routes_df.count())
print("Columns:", routes_df.columns)

display(routes_df.limit(10))

In [ ]:
# Phase 7: Read and Validate All Important Static Datasets
base_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/gtfs_static"

# 1. routes.txt
routes_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{base_path}/routes.txt")
routes_count = routes_df.count()
print(f"routes.txt:     {routes_count:,} rows (Expected ~239)")

# 2. trips.txt
trips_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{base_path}/trips.txt")
trips_count = trips_df.count()
print(f"trips.txt:      {trips_count:,} rows (Expected ~63,023)")

# 3. stops.txt
stops_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{base_path}/stops.txt")
stops_count = stops_df.count()
print(f"stops.txt:      {stops_count:,} rows (Expected ~8,916)")

# 4. stop_times.txt
stop_times_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{base_path}/stop_times.txt")
stop_times_count = stop_times_df.count()
print(f"stop_times.txt: {stop_times_count:,} rows (Expected ~1,844,693)")

In [ ]:
# Phase 8: Validate Data Relationships and Check for Orphan Records
from pyspark.sql.functions import col

# Ensure consistent string comparison for join keys
routes_clean = routes_df.withColumn("route_id", col("route_id").cast("string"))
trips_clean = trips_df.withColumn("trip_id", col("trip_id").cast("string")).withColumn("route_id", col("route_id").cast("string"))
stops_clean = stops_df.withColumn("stop_id", col("stop_id").cast("string"))
stop_times_clean = stop_times_df.withColumn("trip_id", col("trip_id").cast("string")).withColumn("stop_id", col("stop_id").cast("string"))

# 1. trips.route_id -> routes.route_id
orphan_trips_routes = trips_clean.join(routes_clean, on="route_id", how="left_anti")
orphan_trips_routes_cnt = orphan_trips_routes.count()
print(f"Orphan trips (route_id not found in routes):       {orphan_trips_routes_cnt}")

# 2. stop_times.trip_id -> trips.trip_id
orphan_st_trips = stop_times_clean.join(trips_clean, on="trip_id", how="left_anti")
orphan_st_trips_cnt = orphan_st_trips.count()
print(f"Orphan stop_times (trip_id not found in trips):     {orphan_st_trips_cnt}")

# 3. stop_times.stop_id -> stops.stop_id
orphan_st_stops = stop_times_clean.join(stops_clean, on="stop_id", how="left_anti")
orphan_st_stops_cnt = orphan_st_stops.count()
print(f"Orphan stop_times (stop_id not found in stops):     {orphan_st_stops_cnt}")

In [ ]:
# Display Preview of DataFrames
print("--- Routes Sample ---")
routes_df.show(5, truncate=False)

print("--- Trips Sample ---")
trips_df.show(5, truncate=False)

print("--- Stops Sample ---")
stops_df.show(5, truncate=False)

print("--- Stop Times Sample ---")
stop_times_df.show(5, truncate=False)